# Part 3 — Selecting 1,000 compounds for follow-up

**The task.** With a budget of 1,000 purchasable compounds, which would we buy and run through the
CYP3A4 reactivity assay — and, per the prompt, *what do we hope to learn from them?*

That second clause is the whole exercise. The obvious answer — rank purchasable space by predicted
reactivity, buy the top 1,000 — fails it badly, and the reason is visible in our own results.

> **Note on execution.** The scoring pass over 300,000 candidates takes several minutes, so this
> notebook loads the saved scores from `data/processed/zinc_scored.parquet` rather than recomputing
> them. `scripts/run_part3.py` is the reproducible entry point; `make part3` runs it end to end.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from octant_cyp import io, features, models, selection

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

calls = pd.read_csv("../results/reactivity_calls_all.csv")
train = calls[calls.enzyme == "CYP3A4"]
print("what we already know about CYP3A4 in this library:")
print(train.call.value_counts().to_string())
print(f"\nsubstrate rate: {(train.call == 'substrate').mean():.1%}")

what we already know about CYP3A4 in this library:
call
substrate        884
inconclusive     214
non-substrate    125

substrate rate: 72.3%


## 1. Why not simply buy the top 1,000 predicted substrates?

Two numbers settle it:

- **72% of the assayed library are already CYP3A4 substrates.** Base rates are high; this enzyme
  metabolises most drug-like chemistry.
- **The model's scaffold-split ROC-AUC is 0.82.**

So a thousand high-confidence predictions would come back roughly 900 positive. We would have spent
the entire budget to confirm something we already believe, and learned almost nothing — no decision
downstream changes based on that result.

The useful question is not *"which compounds are most likely substrates?"* but **"which purchases
would most change what we know?"** Each bucket below is tied to a stated uncertainty and, crucially,
to what a *surprising* result would mean.

## 2. The candidate pool

The prompt names Enamine and MolPort. Neither can be fetched by script — Enamine's download links
redirect to a request form, MolPort's catalogue sits behind registration — so neither supports a
pipeline that reproduces from a clean clone.

ZINC aggregates exactly those vendors. Selecting from ZINC's **in-stock** tranches (purchasability
codes A/B) therefore yields genuinely Enamine/MolPort-orderable compounds while keeping everything
reproducible, and each row carries a lookup URL resolving to current suppliers and catalogue numbers.

Tranches were chosen to bracket the assayed library rather than by generic drug-likeness: MW bins
F–J (~350–500 Da), logP bins F–J (~2.5–5).

In [2]:
cand = pd.read_parquet("../data/processed/zinc_scored.parquet")
print(f"scored candidates: {len(cand):,}")
print("\nproperty match against the assayed library:")
print(pd.DataFrame({
    "assayed (CYP3A4 library)": [421.5, 3.60],
    "vendor candidates":        [cand.mw.median(), cand.clogp.median()],
}, index=["median MW", "median cLogP"]).round(2).to_string())

scored candidates: 300,000

property match against the assayed library:
              assayed (CYP3A4 library)  vendor candidates
median MW                        421.5             397.48
median cLogP                       3.6               3.56


Closely matched — predictions are being made in the property region the model was trained on.

### A filter we deliberately did *not* apply

The standard move is to run PAINS **and** Brenk. Brenk rejects ~99% of this pool, and more
importantly it rejects **anilines, nitroaromatics and halides** — anilines being the strongest
*depleted* substructure in our own Part 2 results.

Screening those out would remove precisely the compounds able to test our own conclusions. Brenk is
a lead-discovery filter aimed at compounds you would not want to develop; that is a different
question from what you would want to *measure* in a metabolism assay. PAINS is retained, since
assay interference is a real concern here.

## 3. Two models, and why the ranking model is a regressor

**MS-detectability.** A compound that does not ionise yields no depletion measurement at all — the
assay simply fails for it. The OpenADMET write-up is explicit that pre-profiling for MS-compatible
molecules "effectively blinds us to a subset of chemical space," so this is a hard constraint, not a
nicety. Trained on the 11,353-compound `will_it_fly` screen.

The threshold is data-derived rather than assumed: a compound is assay-ready if its ionisation area
reaches **2,824** — the weakest control well that actually supported a reactivity measurement in
this very screen.

In [3]:
det_cv = pd.read_csv("../results/part3_detectability_cv.csv")
fly = io.load_will_it_fly()
y_fly = selection.detectability_labels(fly)
print(f"training compounds: {len(fly):,}   detectable at area >= "
      f"{selection.DETECTABILITY_FLOOR:.0f}: {y_fly.mean():.1%}\n")
print(det_cv.round(3).to_string(index=False))

training compounds: 11,353   detectable at area >= 2824: 90.6%

               model  roc_auc  pr_auc  brier  prevalence     n
 baseline_prevalence    0.489   0.904  0.085       0.906 11353
logistic_descriptors    0.816   0.970  0.104       0.906 11353
       random_forest    0.900   0.986  0.059       0.906 11353


**Ranking model — regression, not classification.** Part 1's offset correction shrank the confident
CYP3A4 negative set to 125, leaving the classifier at 87.6% prevalence and few negatives to learn
from. Regressing the *continuous* log fold-change instead uses all 1,223 compounds, does not depend
on where the substrate boundary was drawn, and produces a ranking — which is what a selection
actually needs.

In [4]:
reg_cv = pd.read_csv("../results/part3_regressor_cv.csv")
print("scaffold-grouped CV on the continuous log10 fold-change:\n")
print(reg_cv.round(3).to_string(index=False))

scaffold-grouped CV on the continuous log10 fold-change:

            model     r2   mae  spearman    n
    baseline_mean -0.004 1.116    -0.089 1223
ridge_descriptors -1.100 1.469     0.327 1223
    random_forest  0.288 0.876     0.540 1223


Spearman 0.54 under scaffold-grouped CV. Modest, and deliberately not oversold — R² of 0.29 means
most variance is unexplained. It ranks usefully; it does not predict precisely. Everything below is
built to respect that.

## 4. Applicability domain — where the model may not be believed

Predictions for compounds unlike anything in training are extrapolation. Rather than pick a round
threshold, define it against the training set's **own internal structure**: how similar are these
1,223 compounds to *each other*?

In [5]:
ad_cut = 0.340  # 10th percentile of training-set internal NN similarity (computed in run_part3.py)
print("training-set internal nearest-neighbour Tanimoto: median 0.676, p10 0.340")
print(f"=> AD cutoff = {ad_cut} (a candidate must be at least as well-supported")
print("   as the most isolated 10% of the training data itself)\n")
print(f"vendor candidates' nearest-neighbour similarity to training: "
      f"median {cand.max_sim_to_training.median():.3f}")
print(f"fraction of the vendor pool inside the domain: "
      f"{(cand.max_sim_to_training >= ad_cut).mean():.1%}")

training-set internal nearest-neighbour Tanimoto: median 0.676, p10 0.340
=> AD cutoff = 0.34 (a candidate must be at least as well-supported
   as the most isolated 10% of the training data itself)

vendor candidates' nearest-neighbour similarity to training: median 0.310
fraction of the vendor pool inside the domain: 31.4%


This is the single most important caveat in Part 3, and it took a wrong turn to state properly.

My first run printed *"outside applicability domain (by design, expansion bucket): 512"* — implying
the overflow was intentional, when that bucket holds only 100 compounds. The real explanation:
**in-stock vendor space is genuinely more distant from this diversity library than the library is
from itself** (median NN 0.35 versus 0.68). Roughly half of any purchasable selection will sit
outside the domain, whatever we do.

That is not a defect in the selection — it is the honest scope limit on every prediction made here,
and it is the strongest argument for funding the expansion bucket at all.

## 5. Budget allocation

Five buckets. The column that matters is the last one.

| Bucket | n | Question it answers | What a surprise would mean |
|---|---|---|---|
| Activity-cliff resolution | 250 | Does a confirmed cliff generalise into new chemistry? | The cliff was local, not a transferable SAR rule |
| Substructure tests | 240 | Are the implicated groups causal or confounded? | Part 2's enrichments are scaffold-driven artefacts |
| Uncertainty sampling | 210 | Where does the model genuinely not know? | Largest expected reduction in future model error |
| Model validation | 200 | Do the predicted values mean what they say? | The model ranks but cannot be used as a probability |
| Space expansion | 100 | Does anything transfer off-domain? | Honest bounds on where the model may be applied |

Two design choices inside these are worth defending.

**Uncertainty sampling uses tree disagreement, not prediction entropy.** Entropy peaks at p=0.5 even
when every tree agrees — that is a compound the model *confidently* believes is borderline, which
teaches nothing. Variance across trees instead marks regions where the training data does not
determine the answer.

**Validation picks span the predicted range rather than the top of it,** including compounds
predicted inactive. Calibration can only be measured where predictions exist, and Part 2 showed the
classifier never predicts below ~0.42 — so the low end is untestable unless we buy it.

In [6]:
sel = pd.read_csv("../results/followup_1000.csv")
print(f"selected: {len(sel)}\n")
print(sel.bucket.value_counts().to_string())
print("\npredicted % remaining per bucket:")
print(sel.groupby("bucket").pct_remaining_pred.describe()[["count", "min", "50%", "max"]]
      .round(1).to_string())

selected: 1000

bucket
cliff_resolution        250
substructure_test       240
uncertainty_sampling    210
model_validation        200
space_expansion         100

predicted % remaining per bucket:
                      count  min  50%   max
bucket                                     
cliff_resolution      250.0  0.1  7.4  69.3
model_validation      200.0  0.1  3.3  81.5
space_expansion       100.0  1.3  9.5  44.3
substructure_test     240.0  0.2  3.1  61.9
uncertainty_sampling  210.0  0.4  1.4   7.4


### A bug worth keeping visible

In the first run the `substructure_test` bucket came back with min = median = max = **3.1%
remaining** — all 240 compounds carrying an identical prediction.

The cause was my own priority function. I wanted picks to *span* the predicted range so the
substructure's effect would not be confounded with predicted activity, but I implemented "closest
to the midpoint of the range", which does the opposite: it collapses the bucket onto a single value.
The bucket would have tested nothing at all.

It now stratifies across five bins, and the spread above (0.2% → 61.9%) shows it working. The lesson
generalises: a bucket that looks suspiciously homogeneous is usually a selection bug rather than a
real property of chemical space.

## 6. What the deliverable carries

Every row is auditable — a reviewer can ask "why this compound?" and get an answer from the file
itself rather than from prose.

In [7]:
print("columns:\n ", "\n  ".join(sel.columns))
print("\nexample rows (one per bucket):")
ex = sel.groupby("bucket").head(1)[
    ["zinc_id", "bucket", "pct_remaining_pred", "pred_uncertainty",
     "p_detectable", "max_sim_to_training", "in_applicability_domain"]]
print(ex.round(3).to_string(index=False))

columns:
  zinc_id
  smiles
  bucket
  pred_log10fc
  pct_remaining_pred
  pred_uncertainty
  p_detectable
  max_sim_to_training
  in_applicability_domain
  max_sim_to_cliff
  mw
  clogp
  tranche
  purchasability
  vendor_lookup
  has_aniline
  has_amide
  has_sulfonamide
  has_tertiary_aliphatic_amine

example rows (one per bucket):
  zinc_id               bucket  pct_remaining_pred  pred_uncertainty  p_detectable  max_sim_to_training  in_applicability_domain
 12634208     model_validation               0.111             1.043         0.874                0.412                     True
  9831122     cliff_resolution               0.799             1.199         0.951                0.881                     True
 58138070    substructure_test               0.235             1.383         0.964                0.342                     True
 57996484 uncertainty_sampling               2.747             1.634         0.646                0.389                     True
253595932      spa

In [8]:
n_out = int((~sel.in_applicability_domain).sum())
print(f"outside applicability domain : {n_out} of {len(sel)} ({100*n_out/len(sel):.0f}%)")
print(f"median NN similarity to training: {sel.max_sim_to_training.median():.3f}")
print(f"all predicted MS-detectable  : {bool((sel.p_detectable >= 0.5).all())}")
print(f"unique compounds             : {sel.zinc_id.nunique()} (no duplicates)")
print(f"\nexample vendor lookup: {sel.vendor_lookup.iloc[0]}")

outside applicability domain : 470 of 1000 (47%)
median NN similarity to training: 0.348
all predicted MS-detectable  : True
unique compounds             : 1000 (no duplicates)

example vendor lookup: https://zinc20.docking.org/substances/12634208


## 7. What we would actually learn

Honest expectations, bucket by bucket:

- **Cliff resolution (250).** If neighbours of a confirmed cliff land on the predicted side, the
  cliff encodes a transferable rule and belongs in a design guideline. If they scatter, it was a
  property of those two molecules and should not be generalised.
- **Substructure tests (240).** The most likely bucket to *overturn* a Part 2 conclusion, which is
  exactly why it is funded. Enrichment is correlational; anilines co-occur with particular scaffolds
  and property ranges, so the association may not be causal.
- **Uncertainty sampling (210).** Should give the largest improvement in the next model iteration,
  and will not look impressive as a hit rate — by construction these are coin-flips.
- **Model validation (200).** Tells us whether the predicted numbers can drive decisions or only
  rank. Part 2 already showed the model is under-confident, so I expect observed rates to exceed
  predicted ones.
- **Space expansion (100).** Likely the lowest hit rate of the five, and the most informative about
  where the model stops working.

**The honest bottom line.** A single-timepoint depletion screen tells us *whether* a compound is
turned over, not how fast. Nothing here estimates intrinsic clearance, and roughly half of what we
would buy sits outside the model's applicability domain. The selection is designed so that a
disappointing result is still an informative one — which, on a 1,000-compound budget, is the
property worth optimising for.